# Tratamento da Base de Dados

Nesta etapa, a base original será preparada para as análises seguintes.
O tratamento consiste em corrigir a leitura dos dados, transformar os valores em formato numérico, remover linhas que não representam contas e padronizar os textos.
Após o tratamento, serão feitas algumas verificações para confirmar a estrutura da base.

## 1. IMPORTANDO BIBLIOTECAS

In [1]:
import pandas as pd
import numpy as np

pd.set_option(
    "display.float_format",
    lambda x: f"{x:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".")
)

## 2. Leitura da base original

In [2]:
caminho = "../data/raw/Demonstrativo Fecap v3.csv"

df = pd.read_csv(
    caminho,
    sep=";",
    encoding="latin-1",
    header=None
)

df.columns = ["Ano", "Cenario", "Conta", "Valor"]

print("Dimensões da base:", df.shape)
display(df.head(10))

Dimensões da base: (1002000, 4)


,Ano,Cenario,Conta,Valor
0,Ano 1,Total Cen_00001,NaN,"0,025138815"
1,Ano 1,Total Cen_00001,BAL - Total do Ativo,"10.631.127.075,26"
2,Ano 1,Total Cen_00001,BAL - Ativo Circulante,"1.795.720.872,18"
3,Ano 1,Total Cen_00001,BAL - Disponível,"1.205.264.965,52"
4,Ano 1,Total Cen_00001,BAL - Contas a Receber - SWAP,"95.680.421,92"
5,Ano 1,Total Cen_00001,BAL - Contas a Receber - Partes Relacionadas,"584.661,47"
6,Ano 1,Total Cen_00001,BAL - Contas a Receber - Clientes,"453.870.957,15"
7,Ano 1,Total Cen_00001,BAL - Estoques Diversos,"86.620,79"
8,Ano 1,Total Cen_00001,BAL - Outros Créditos,"40.233.245,33"
9,Ano 1,Total Cen_00001,BAL - Realizável a Longo Prazo,"639.023.858,23"


## 3. Verificação inicial

Antes de alterar os dados, verificamos os tipos das colunas, os valores ausentes e a quantidade de registros distintos.

In [3]:
print("Tipos das colunas:")
display(df.dtypes)

print("\nValores ausentes:")
display(df.isna().sum())

print("\nQuantidade de valores distintos:")
display(df.nunique())

Tipos das colunas:


Ano        str
Cenario    str
Conta      str
Valor      str
dtype: object


Valores ausentes:


Ano            0
Cenario        0
Conta      43200
Valor          0
dtype: int64


Quantidade de valores distintos:


Ano            12
Cenario      1200
Conta          69
Valor      443175
dtype: int64

## 4. Conversão dos valores

Os valores financeiros estão armazenados como texto e utilizam o padrão brasileiro de formatação.

A conversão remove os pontos usados como separadores de milhar e troca a vírgula decimal por ponto.

In [4]:
df["Valor_numerico"] = (
    df["Valor"]
    .str.replace(".", "", regex=False)
    .str.replace(",", ".", regex=False)
    .astype(float)
)

print("Valores convertidos:", df["Valor_numerico"].notna().sum())
print("Valores que não puderam ser convertidos:", df["Valor_numerico"].isna().sum())

Valores convertidos: 1002000
Valores que não puderam ser convertidos: 0


## 5. Remoção das linhas sem conta

Foram identificadas linhas sem informação na coluna `Conta`.

Essas linhas não representam contas contábeis e aparecem como separadores estruturais da base. Por isso, serão removidas.

In [5]:
linhas_sem_conta = df["Conta"].isna().sum()

df_tratado = df[df["Conta"].notna()].copy()

print("Linhas sem conta removidas:", linhas_sem_conta)
print("Dimensões após a remoção:", df_tratado.shape)

Linhas sem conta removidas: 43200
Dimensões após a remoção: (958800, 5)


## 6. Padronização dos textos

Os campos de texto são padronizados para evitar diferenças causadas por espaços desnecessários.


In [6]:
df_tratado["Ano"] = df_tratado["Ano"].str.strip()

df_tratado["Cenario"] = df_tratado["Cenario"].str.strip()

df_tratado["Conta"] = (
    df_tratado["Conta"]
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

print("Padronização concluída.")
display(df_tratado.head())

Padronização concluída.


,Ano,Cenario,Conta,Valor,Valor_numerico
1,Ano 1,Total Cen_00001,BAL - Total do Ativo,"10.631.127.075,26","10.631.127.075,26"
2,Ano 1,Total Cen_00001,BAL - Ativo Circulante,"1.795.720.872,18","1.795.720.872,18"
3,Ano 1,Total Cen_00001,BAL - Disponível,"1.205.264.965,52","1.205.264.965,52"
4,Ano 1,Total Cen_00001,BAL - Contas a Receber - SWAP,"95.680.421,92","95.680.421,92"
5,Ano 1,Total Cen_00001,BAL - Contas a Receber - Partes Relacionadas,"584.661,47","584.661,47"


## 7. Verificação da base tratada

Após as alterações, verificamos se ainda existem valores ausentes ou contas duplicadas dentro de um mesmo cenário e ano.

In [7]:
print("Valores ausentes:")
display(df_tratado.isna().sum())

duplicados = (
    df_tratado
    .duplicated(subset=["Cenario", "Ano", "Conta"])
    .sum()
)

print("Contas duplicadas dentro de Cenario + Ano:", duplicados)
print("Quantidade de contas distintas:", df_tratado["Conta"].nunique())

Valores ausentes:


Ano               0
Cenario           0
Conta             0
Valor             0
Valor_numerico    0
dtype: int64

Contas duplicadas dentro de Cenario + Ano: 0
Quantidade de contas distintas: 69


## 8. Quantidade de contas por ano

A quantidade de contas não é igual em todos os anos.

Essa verificação permite identificar a estrutura da base.

In [8]:
contas_por_ano = (
    df_tratado
    .groupby("Ano")["Conta"]
    .nunique()
    .reset_index(name="Quantidade_Contas")
)

display(contas_por_ano)

,Ano,Quantidade_Contas
0,Ano 1,68
1,Ano 10,69
2,Ano 11,69
3,Ano 12,42
4,Ano 2,68
5,Ano 3,69
6,Ano 4,69
7,Ano 5,69
8,Ano 6,69
9,Ano 7,69


## 9. Contas por cenário e ano

Verificamos quantas contas existem em cada combinação de cenário e ano.

In [9]:
contagem_contas = (
    df_tratado
    .groupby(["Cenario", "Ano"])["Conta"]
    .nunique()
    .reset_index(name="Quantidade_Contas")
)

display(contagem_contas.head(20))

,Cenario,Ano,Quantidade_Contas
0,Total Cen_00001,Ano 1,68
1,Total Cen_00001,Ano 10,69
2,Total Cen_00001,Ano 11,69
3,Total Cen_00001,Ano 12,42
4,Total Cen_00001,Ano 2,68
5,Total Cen_00001,Ano 3,69
6,Total Cen_00001,Ano 4,69
7,Total Cen_00001,Ano 5,69
8,Total Cen_00001,Ano 6,69
9,Total Cen_00001,Ano 7,69


## 10. Comparação das contas entre os anos

A quantidade de contas varia ao longo dos anos.  
Aqui verificamos quais contas aparecem ou deixam de aparecer em cada período.

In [10]:
# Lista de contas existentes em cada ano

contas_por_ano = {
    ano: set(df_tratado.loc[df_tratado["Ano"] == ano, "Conta"].unique())
    for ano in sorted(df_tratado["Ano"].unique())
}

anos = sorted(contas_por_ano.keys())

for i in range(len(anos) - 1):
    ano_atual = anos[i]
    proximo_ano = anos[i + 1]

    adicionadas = sorted(contas_por_ano[proximo_ano] - contas_por_ano[ano_atual])
    removidas = sorted(contas_por_ano[ano_atual] - contas_por_ano[proximo_ano])

    print(f"{ano_atual} → {proximo_ano}")
    print(f"Contas adicionadas: {len(adicionadas)}")
    for conta in adicionadas:
        print(f"  + {conta}")

    print(f"Contas removidas: {len(removidas)}")
    for conta in removidas:
        print(f"  - {conta}")

    print("-" * 60)

Ano 1 → Ano 10
Contas adicionadas: 1
  + BAL - Dividendos Antecipados
Contas removidas: 0
------------------------------------------------------------
Ano 10 → Ano 11
Contas adicionadas: 0
Contas removidas: 0
------------------------------------------------------------
Ano 11 → Ano 12
Contas adicionadas: 0
Contas removidas: 27
  - BAL - Amortização Acumulada
  - BAL - Capital Social
  - BAL - Contas a Pagar - Parte Relacionada
  - BAL - Contas a Receber - Partes Relacionadas
  - BAL - Contas a Receber - SWAP
  - BAL - Créditos Tributários
  - BAL - Diferido
  - BAL - Dividendos Antecipados
  - BAL - Emprést
  - BAL - Encargos Sociais e Trabalhistas
  - BAL - Estoques Diversos
  - BAL - Exigível a Longo Prazo
  - BAL - Fornecedores
  - BAL - Obrigações com o Poder Concedente
  - BAL - Outorga da Concessão
  - BAL - Outros Créditos LP
  - BAL - Outros Débitos
  - BAL - Outros deb
  - BAL - Patrimônio Líquido
  - BAL - Prov para Contingências
  - BAL - Provisão Manutenção
  - BAL - Reserv

## 11. Resumo das contas que mudam ao longo dos anos

In [11]:
# Resumo das mudanças na estrutura das contas

mudancas = []

for i in range(len(anos) - 1):
    ano_atual = anos[i]
    proximo_ano = anos[i + 1]

    adicionadas = contas_por_ano[proximo_ano] - contas_por_ano[ano_atual]
    removidas = contas_por_ano[ano_atual] - contas_por_ano[proximo_ano]

    if adicionadas or removidas:
        mudancas.append({
            "De": ano_atual,
            "Para": proximo_ano,
            "Contas adicionadas": ", ".join(sorted(adicionadas)) if adicionadas else "Nenhuma",
            "Contas removidas": ", ".join(sorted(removidas)) if removidas else "Nenhuma"
        })

df_mudancas = pd.DataFrame(mudancas)

display(df_mudancas)

,De,Para,Contas adicionadas,Contas removidas
0,Ano 1,Ano 10,BAL - Dividendos Antecipados,Nenhuma
1,Ano 11,Ano 12,Nenhuma,"BAL - Amortização Acumulada, BAL - Capital Soc..."
2,Ano 12,Ano 2,"BAL - Amortização Acumulada, BAL - Capital Soc...",Nenhuma
3,Ano 2,Ano 3,BAL - Dividendos Antecipados,Nenhuma


## 12. Verificação dos cenários

Verificamos se todos os cenários possuem os 12 anos disponíveis.

In [12]:
anos_por_cenario = (
    df_tratado
    .groupby("Cenario")["Ano"]
    .nunique()
)

print("Quantidade de cenários:", anos_por_cenario.size)
print("Cenários com 12 anos:", (anos_por_cenario == 12).sum())
print("Cenários com menos de 12 anos:", (anos_por_cenario < 12).sum())

Quantidade de cenários: 1200
Cenários com 12 anos: 1200
Cenários com menos de 12 anos: 0


## 13. Verificação das demonstrações

As contas são identificadas pelos prefixos `BAL`, `DRE` e `FLU`.

Verificamos a quantidade de registros de cada demonstração para confirmar a estrutura da base.

In [13]:
demonstracoes = (
    df_tratado["Conta"]
    .str[:3]
    .value_counts()
    .sort_index()
)

display(demonstracoes)

Conta
BAL    584400
DRE    201600
FLU    172800
Name: count, dtype: int64

## 14. Conferência do Balanço Patrimonial

Verificamos se o Total do Ativo e o Total do Passivo apresentam valores compatíveis em cada cenário e ano.

In [14]:
bp = (
    df_tratado[
        df_tratado["Conta"].isin([
            "BAL - Total do Ativo",
            "BAL - Total do Passivo"
        ])
    ]
    .pivot_table(
        index=["Cenario", "Ano"],
        columns="Conta",
        values="Valor_numerico"
    )
)

bp["Diferenca"] = (
    bp["BAL - Total do Ativo"] +
    bp["BAL - Total do Passivo"]
)

print("Combinações Cenario + Ano:", len(bp))
print("Maior diferença absoluta encontrada:")

display(
    bp["Diferenca"]
    .abs()
    .max()
)

Combinações Cenario + Ano: 14400
Maior diferença absoluta encontrada:


np.float64(0.030000686645507812)

## 15. Base final

Após o tratamento, mantemos apenas as informações necessárias para as próximas etapas.

In [15]:
base_final = (
    df_tratado[
        ["Cenario", "Ano", "Conta", "Valor_numerico"]
    ]
    .sort_values(["Cenario", "Ano", "Conta"])
    .reset_index(drop=True)
)

print("Dimensões da base final:", base_final.shape)
display(base_final.head(10))

Dimensões da base final: (958800, 4)


,Cenario,Ano,Conta,Valor_numerico
0,Total Cen_00001,Ano 1,BAL - Amortização - Intangível,"-5.047.123.846,36"
1,Total Cen_00001,Ano 1,BAL - Amortização Acumulada,"-30.111.275,81"
2,Total Cen_00001,Ano 1,BAL - At Fiscal Diferido,"-103.969.289,50"
3,Total Cen_00001,Ano 1,BAL - Ativo Circulante,"1.795.720.872,18"
4,Total Cen_00001,Ano 1,BAL - Capital Social,"-309.618.939,00"
5,Total Cen_00001,Ano 1,BAL - Contas a Pagar - Parte Relacionada,"-14.086.270,70"
6,Total Cen_00001,Ano 1,BAL - Contas a Receber - Clientes,"453.870.957,15"
7,Total Cen_00001,Ano 1,BAL - Contas a Receber - Partes Relacionadas,"584.661,47"
8,Total Cen_00001,Ano 1,BAL - Contas a Receber - SWAP,"95.680.421,92"
9,Total Cen_00001,Ano 1,BAL - Créditos Tributários,"1.846.452,68"


## 16. Verificação dos sinais por conta

Nesta etapa, verificamos se os valores positivos e negativos estão coerentes com cada conta.

A análise serve para identificar possíveis inconsistências na estrutura dos dados antes do cálculo dos indicadores.

In [16]:
sinais_por_conta = (
    df_tratado
    .groupby("Conta")["Valor_numerico"]
    .agg(
        Total="count",
        Positivos=lambda x: (x > 0).sum(),
        Negativos=lambda x: (x < 0).sum(),
        Zerados=lambda x: (x == 0).sum()
    )
    .reset_index()
)

sinais_por_conta["% Positivos"] = (
    sinais_por_conta["Positivos"] / sinais_por_conta["Total"] * 100
)

sinais_por_conta["% Negativos"] = (
    sinais_por_conta["Negativos"] / sinais_por_conta["Total"] * 100
)

display(sinais_por_conta)

,Conta,Total,Positivos,Negativos,Zerados,% Positivos,% Negativos
0,BAL - Amortização - Intangível,14400,0,14400,0,"0,00","100,00"
1,BAL - Amortização Acumulada,13200,0,13200,0,"0,00","100,00"
2,BAL - At Fiscal Diferido,14400,13200,1200,0,"91,67","8,33"
3,BAL - Ativo Circulante,14400,14378,22,0,"99,85","0,15"
4,BAL - Capital Social,13200,0,13200,0,"0,00","100,00"
...,...,...,...,...,...,...,...
64,FLU - Receita,14400,14400,0,0,"100,00","0,00"
65,FLU - Resultado Financeiro,14400,402,13998,0,"2,79","97,21"
66,FLU - Saldo Final,14400,14399,1,0,"99,99","0,01"
67,FLU - Saldo Inicial,14400,14400,0,0,"100,00","0,00"


# 17. Contas que possuem valores positivos e negativos

In [17]:
contas_mistos = sinais_por_conta[
    (sinais_por_conta["Positivos"] > 0) &
    (sinais_por_conta["Negativos"] > 0)
]

print("Contas que apresentam valores positivos e negativos:")
display(contas_mistos)

Contas que apresentam valores positivos e negativos:


,Conta,Total,Positivos,Negativos,Zerados,% Positivos,% Negativos
2,BAL - At Fiscal Diferido,14400,13200,1200,0,"91,67","8,33"
3,BAL - Ativo Circulante,14400,14378,22,0,"99,85","0,15"
6,BAL - Contas a Receber - Clientes,14400,13200,1200,0,"91,67","8,33"
10,BAL - Depreciação Acumulada,14400,1200,13200,0,"8,33","91,67"
12,BAL - Disponível,14400,14399,1,0,"99,99","0,01"
15,BAL - Empréstimos,14400,9600,4800,0,"66,67","33,33"
21,BAL - Investimentos - Imobilizado,14400,14368,27,5,"99,78","0,19"
25,BAL - Outros Créditos,14400,2400,12000,0,"16,67","83,33"
29,BAL - Passivo Circulante,14400,8966,5434,0,"62,26","37,74"
30,BAL - Patrimônio Líquido,13200,63,13137,0,"0,48","99,52"


## 18. Salvando a base tratada

A base tratada será salva para ser utilizada nos próximos notebooks, evitando repetir o tratamento da base original.

In [18]:
caminho_saida = "../staging/dados_tratados.csv"

base_final.to_csv(
    caminho_saida,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

print(f"Base tratada salva em: {caminho_saida}")

Base tratada salva em: ../staging/dados_tratados.csv


In [19]:
base_verificacao = pd.read_csv(
    "../staging/dados_tratados.csv",
    sep=";",
    encoding="utf-8-sig"
)

print("Dimensões:", base_verificacao.shape)
print("\nColunas:")
print(base_verificacao.columns.tolist())

print("\nValores nulos:")
print(base_verificacao.isna().sum())

print("\nQuantidade de cenários:", base_verificacao["Cenario"].nunique())
print("Quantidade de anos:", base_verificacao["Ano"].nunique())
print("Quantidade de contas:", base_verificacao["Conta"].nunique())

display(base_verificacao.head(10))

Dimensões: (958800, 4)

Colunas:
['Cenario', 'Ano', 'Conta', 'Valor_numerico']

Valores nulos:
Cenario           0
Ano               0
Conta             0
Valor_numerico    0
dtype: int64

Quantidade de cenários: 1200
Quantidade de anos: 12
Quantidade de contas: 69


,Cenario,Ano,Conta,Valor_numerico
0,Total Cen_00001,Ano 1,BAL - Amortização - Intangível,"-5.047.123.846,36"
1,Total Cen_00001,Ano 1,BAL - Amortização Acumulada,"-30.111.275,81"
2,Total Cen_00001,Ano 1,BAL - At Fiscal Diferido,"-103.969.289,50"
3,Total Cen_00001,Ano 1,BAL - Ativo Circulante,"1.795.720.872,18"
4,Total Cen_00001,Ano 1,BAL - Capital Social,"-309.618.939,00"
5,Total Cen_00001,Ano 1,BAL - Contas a Pagar - Parte Relacionada,"-14.086.270,70"
6,Total Cen_00001,Ano 1,BAL - Contas a Receber - Clientes,"453.870.957,15"
7,Total Cen_00001,Ano 1,BAL - Contas a Receber - Partes Relacionadas,"584.661,47"
8,Total Cen_00001,Ano 1,BAL - Contas a Receber - SWAP,"95.680.421,92"
9,Total Cen_00001,Ano 1,BAL - Créditos Tributários,"1.846.452,68"
